# Twitter Sentiment Analysis — ML Models vs SimpleRNN

This notebook works on the **Twitter Entity Sentiment Analysis** dataset (`twitter_training.csv`).

**Pipeline**
1. Load & explore the data
2. Clean & preprocess tweet text
3. Train several classical ML models (TF-IDF features)
4. Train a **SimpleRNN** deep learning model (target: **accuracy > 80%**)
5. Compare all models with accuracy bar charts

> Upload `twitter_training.csv` when prompted in the **Load Data** cell (unzip `twitter_training_csv.zip` first if needed).


## 1. Install & Import Libraries

In [ ]:
# If running on a fresh Colab runtime, these are usually pre-installed.
# Uncomment if any import below fails.
# !pip install -q scikit-learn tensorflow pandas matplotlib seaborn nltk

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")
print("TensorFlow version:", tf.__version__)


## 2. Load Data

In [ ]:
# Try to locate the CSV. If it's not found, upload it (from the unzipped
# twitter_training_csv.zip) using the Colab file picker.
import os

CSV_PATH = "twitter_training.csv"

if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print("Please upload 'twitter_training.csv' (unzip twitter_training_csv.zip first if needed):")
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            "twitter_training.csv not found. Place it in the working directory."
        )

# The dataset has no header row: columns are id, entity, sentiment, text
df = pd.read_csv(CSV_PATH, header=None, names=["id", "entity", "sentiment", "text"])
print("Shape:", df.shape)
df.head()


## 3. Explore the Data

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nSentiment class counts:\n", df['sentiment'].value_counts())


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="sentiment",
              order=df["sentiment"].value_counts().index,
              palette="viridis")
plt.title("Distribution of Sentiment Classes")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()


## 4. Text Cleaning & Preprocessing

Steps: drop missing tweets, lowercase, remove URLs/mentions/hashtags symbols,
remove punctuation & digits, collapse whitespace, drop tweets that become empty.


In [ ]:
df = df.dropna(subset=["text"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["text", "sentiment"]).reset_index(drop=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)          # URLs
    text = re.sub(r"@\w+", " ", text)                        # mentions
    text = re.sub(r"#", " ", text)                            # hashtag symbol (keep word)
    text = re.sub(r"[^a-z\s]", " ", text)                     # punctuation & digits
    text = re.sub(r"\s+", " ", text).strip()                  # extra whitespace
    return text

df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print("Shape after cleaning:", df.shape)
df[["text", "clean_text", "sentiment"]].head()


In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["sentiment"])
print(dict(zip(le.classes_, le.transform(le.classes_))))


## 5. Train / Test Split

The **same split** (on `clean_text` / `label`) is reused for both the ML models and the SimpleRNN so accuracies are directly comparable.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.2, random_state=SEED, stratify=df["label"]
)

print("Train size:", X_train_text.shape[0])
print("Test size:", X_test_text.shape[0])


## 6. Machine Learning Models (TF-IDF features)

We vectorize the cleaned text with TF-IDF and train four classical models:
Logistic Regression, Multinomial Naive Bayes, Linear SVM, and Random Forest.


In [ ]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)


In [ ]:
ml_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
}

ml_results = {}

for name, model in ml_models.items():
    print(f"Training {name} ...")
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    ml_results[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=le.classes_))
    print("-" * 70)


In [ ]:
ml_results_df = pd.DataFrame(list(ml_results.items()), columns=["Model", "Accuracy"]) \
                   .sort_values("Accuracy", ascending=False).reset_index(drop=True)
ml_results_df


## 7. Deep Learning Model — SimpleRNN

Target: **test accuracy > 80%**.

Steps: tokenize text into integer sequences, pad to a fixed length, feed into an
`Embedding -> SimpleRNN -> Dropout -> Dense(softmax)` network.


In [ ]:
VOCAB_SIZE = 20000
MAX_LEN = 40
EMBED_DIM = 128

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

num_classes = len(le.classes_)
print("Vocabulary size used:", min(VOCAB_SIZE, len(tokenizer.word_index) + 1))
print("Number of classes:", num_classes)
print("Padded train shape:", X_train_pad.shape)


In [ ]:
rnn_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    SimpleRNN(128, return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax"),
])

rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

rnn_model.summary()


In [ ]:
early_stop = EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)

history = rnn_model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1,
)


In [ ]:
rnn_loss, rnn_accuracy = rnn_model.evaluate(X_test_pad, y_test, verbose=0)
print(f"SimpleRNN Test Accuracy: {rnn_accuracy:.4f}")
print(f"SimpleRNN Test Loss: {rnn_loss:.4f}")

if rnn_accuracy > 0.80:
    print("\n✅ Target achieved: accuracy is above 80%!")
else:
    print("\n⚠️ Accuracy is below 80% — consider more epochs, a larger MAX_LEN/VOCAB_SIZE, or more RNN units.")


In [ ]:
rnn_preds = np.argmax(rnn_model.predict(X_test_pad), axis=1)
print(classification_report(y_test, rnn_preds, target_names=le.classes_))

cm = confusion_matrix(y_test, rnn_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("SimpleRNN — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["accuracy"], label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[0].axhline(y=0.80, color="red", linestyle="--", label="80% target")
axes[0].set_title("SimpleRNN — Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Validation Loss")
axes[1].set_title("SimpleRNN — Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Accuracy Comparison — ML Models vs SimpleRNN

In [ ]:
all_results = dict(ml_results)
all_results["SimpleRNN"] = rnn_accuracy

results_df = pd.DataFrame(list(all_results.items()), columns=["Model", "Accuracy"]) \
               .sort_values("Accuracy", ascending=False).reset_index(drop=True)
results_df


In [ ]:
plt.figure(figsize=(9, 6))
colors = ["#d62728" if m == "SimpleRNN" else "#1f77b4" for m in results_df["Model"]]
bars = plt.bar(results_df["Model"], results_df["Accuracy"], color=colors)

plt.axhline(y=0.80, color="green", linestyle="--", label="80% target")
plt.ylim(0, 1.0)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison: ML Models vs SimpleRNN")
plt.xticks(rotation=20, ha="right")
plt.legend()

for bar, acc in zip(bars, results_df["Accuracy"]):
    plt.text(bar.get_x() + bar.get_width() / 2, acc + 0.01, f"{acc:.3f}",
              ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(results_df["Accuracy"], labels=results_df["Model"], autopct="%1.1f%%",
        startangle=90, colors=sns.color_palette("viridis", len(results_df)))
plt.title("Relative Accuracy Share Across Models")
plt.show()


## 9. Conclusion

- Several classical ML models (Logistic Regression, Naive Bayes, Linear SVM, Random Forest)
  were trained on TF-IDF features.
- A **SimpleRNN** deep learning model was trained on tokenized/padded sequences and achieved
  a test accuracy printed above (target: **> 80%**).
- The bar chart in Section 8 summarizes and compares the accuracy of every model.

Feel free to tune hyperparameters (`MAX_LEN`, `VOCAB_SIZE`, RNN units, epochs, TF-IDF `max_features`)
to push accuracy even higher.
